# PD detection from speech on PC-GITA — full pipeline on Google Colab

Runtime: **Runtime → Change runtime type → T4 GPU**.

You need your own copy of PC-GITA (it cannot be redistributed) in Google Drive, with the original folder layout:
```
<PCGITA_ROOT>/read text/ayerfuialmedico/sin normalizar/{hc,pd}/*.wav
<PCGITA_ROOT>/monologue/sin normalizar/{hc,pd}/*.wav
```
The main model (5 seeds) takes roughly 15–20 minutes on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PCGITA_ROOT = "/content/drive/MyDrive/Diploma_Spanish"                       # <- edit if needed
METADATA    = f"{PCGITA_ROOT}/Copia de PCGITA_metadata.xlsx"                  # optional
OUTPUTS     = "/content/drive/MyDrive/PD-Detection-PCGITA-outputs"            # runs are saved here (resumable)

In [ ]:
!git clone https://github.com/GeorgeTheo7/PD-Detection-PCGITA.git /content/PD-Detection-PCGITA
%cd /content/PD-Detection-PCGITA
!pip install -q -r requirements.txt

## 1. Preprocess: select tasks, 16 kHz mono, 5 s chunks

In [ ]:
!python main.py prepare --raw-dir "$PCGITA_ROOT" --metadata "$METADATA" --out-dir data/processed

## 2. Cache frozen SSL features (~20–40 s per model)
Use `--ssl xlsr53` for the main model only, or `--ssl all` to also run the ablations.

In [ ]:
!python main.py extract --processed-dir data/processed --features-dir data/features --ssl all

## 3. Nested speaker-independent 5-fold CV (5 seeds)
Main model only: `--experiments coattn_xlsr53`. All ablation variants: `--experiments all` (roughly 2–3 hours on a T4).
If Colab disconnects, re-run the cells above and this one: finished folds are skipped.

In [ ]:
!python main.py train --processed-dir data/processed --features-dir data/features \
    --runs-dir "$OUTPUTS/runs" --experiments coattn_xlsr53

## 4. Evaluate

In [ ]:
!python main.py evaluate --runs-dir "$OUTPUTS/runs" --experiment coattn_xlsr53 --seed 42
# After `--experiments all`:
# !python main.py ablations --runs-dir "$OUTPUTS/runs"

In [ ]:
from IPython.display import Image, display
display(Image(f"{OUTPUTS}/runs/coattn_xlsr53/seed42/confusion_matrix.png", width=380))
display(Image(f"{OUTPUTS}/runs/coattn_xlsr53/seed42/roc_curve.png", width=420))